<a href="https://www.kaggle.com/code/valeriislynko/fruits-classification-hw1-2?scriptVersionId=359148245" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **Завдання 1 (29.09.26):**
* Ознайомитись з даними та структурою даних
* Створіть доповнення (*transform*) для тренувальних та тестових даних. посилання
* Створіть *train_dataset* та *test_dataset* за допомогою **ImageFolder** (папка train)
* Переконайтесь що у вас привильні назви класів
* Створіть **DataLoader** для тренувальних та тестових даних
* Візуалізуйте дані
* Збережіть *kaggle notebook* для подальшої роботи

# * Ознайомлення з даними та структурою даних

In [3]:
# 1. Імпорт бібліотек
import torch                                    # для роботи з нейромережами
from torchvision.datasets import ImageFolder    # для завантаження картинки з папок

In [4]:
# 2. Головна папка з тренувальними даними (шлях до датасету "Fruit Classification")
base_fruits_dataset = "/kaggle/input/datasets/sshikamaru/fruit-recognition/train/train"

# Створюємо датасет  за допомогою ImageFolder - сортує зображення по папках-класах
dataset = ImageFolder(base_fruits_dataset)

In [5]:
# 3. Перевірка кількості зображень у датасеті
len(dataset)

16854

In [6]:
# Список класів(категорій) зображень - назви фруктів та овочів
dataset.classes

['Apple Braeburn',
 'Apple Granny Smith',
 'Apricot',
 'Avocado',
 'Banana',
 'Blueberry',
 'Cactus fruit',
 'Cantaloupe',
 'Cherry',
 'Clementine',
 'Corn',
 'Cucumber Ripe',
 'Grape Blue',
 'Kiwi',
 'Lemon',
 'Limes',
 'Mango',
 'Onion White',
 'Orange',
 'Papaya',
 'Passion Fruit',
 'Peach',
 'Pear',
 'Pepper Green',
 'Pepper Red',
 'Pineapple',
 'Plum',
 'Pomegranate',
 'Potato Red',
 'Raspberry',
 'Strawberry',
 'Tomato',
 'Watermelon']

In [7]:
# зберігаємо мітку у змінну label і виводимо її
label = "Перевірка міток класів"
print(label)

Перевірка міток класів


In [8]:
# Отримуємо 100-те зображення з датасету (Х -картинка, у -мітка/номер класу)
X, y = dataset[100]

# виводимо мітку класу для цього зображення
print(f"Номер класу (label): {y}")
print(f"Назва класу: {dataset.classes[y]}")



Номер класу (label): 0
Назва класу: Apple Braeburn


## * Створюємо доповнення (transform) для тренувальних та тестових даних. посилання

In [9]:
# Створення ТРЕНУВАЛЬНОГО та ТЕСТОВОГО трансформаторів
# Імпортуємо модуль для трансформації зображень
from torchvision import transforms

# Створюємо трансформації:
# Трансформації для тренувальних даних: змінюємо розмір, поворот, дзеркалення та перетворюємо на Tensor (тензор)
train_transforms = transforms.Compose([
    transforms.Resize([100, 100]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

# Трансформації для тестових даних (тільки зміна розміру та тензор)
test_transforms = transforms.Compose([
    transforms.Resize([100, 100]),
    transforms.ToTensor()
])

print("Трансформатори train_transforms та test_transforms успішно створено")

Трансформатори train_transforms та test_transforms успішно створено


# * Створюємо 'train_dataset' та 'test_dataset' за допомогою ImageFolder (папка train)

In [10]:
# 1. ПОДІЛ ДАТАСЕТУ на train(80%) та test(20%)
from torch.utils.data import random_split

# випадково ділимо наш загальний датасет на тренувальну(80%) та тестову(20%) частини
train_dataset, test_dataset = random_split(dataset, [0.8, 0.2])

print(f"Кількість зображень у 'train_dataset': {len(train_dataset)}")
print(f"Кількість зображень у 'test_dataset': {len(test_dataset)}")

Кількість зображень у 'train_dataset': 13484
Кількість зображень у 'test_dataset': 3370


In [11]:
# 2. СТВОРЕННЯ власного ДАТАСЕТУ TransfomDatasetFruits (клас потрібен для застосування трансформацій до картинок при їх отримання)
from torch.utils.data import Dataset

# створюємо власний клас для обгортки датасету з трансформаціями
class TransformDatasetFruits(Dataset):
    def __init__(self, dataset, transformer):
        self.dataset = dataset
        self.transformer = transformer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # отримуємо картинку та мітку з оригінального датасету
        image, label = self.dataset[idx]

        # Якщо трансформер заданий, застосовуємо його до картинки
        if self.transformer:
            image = self.transformer(image)
        return image, label

print("Клас TransformDatasetFruits успішно оголошено")

Клас TransformDatasetFruits успішно оголошено


In [12]:
# 4. ЗАСТОСУВАННЯ КЛАСУ до наших train та test наборів
train_dataset = TransformDatasetFruits(train_dataset, train_transforms)
test_dataset = TransformDatasetFruits(test_dataset, test_transforms)

print("Трансформації успішно застосовано до 'train' та 'test' датасетів!")


Трансформації успішно застосовано до 'train' та 'test' датасетів!


# * Створюємо DataLoader для ТРЕНУВАЛЬНИХ та ТЕСТОВИХ даних

In [13]:
# Крок 1: Створення DataLoader

# НЕЙРОМЕРЕЖА під час навчання опрацьовує дані не по одному файлу, а пачками (батчами-'batch').
# Клас DataLoader створює саме такі пачки/батчі, а для тренувального набору ще й перемішує їх, щоб модель
# не запам'ятовувала порядок, а реально вчилася розпізнавати фрукти/овочі.

# Імпортуємо модуль для завантажувача даних (DataLoader)
from torch.utils.data import DataLoader

batch_size = 64

# Створюємо завантажувачі даних для тренувального та тестового наборів
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"DataLoader успішно створено!")
print(f"Кількість батчів у train_loader: {len(train_loader)}")
print(f"Кількість батчів у test_loader: {len(test_loader)}")


DataLoader успішно створено!
Кількість батчів у train_loader: 211
Кількість батчів у test_loader: 53


In [14]:
# Крок 2: Перевірка форми тензора
# Візьмемо один батч, щоб перевірити розмірність картинок (img.shape або x.shape) перед подачею в нейромережу.

# Отримуємо один батч даних для перевірки форми
for img, labels in train_loader:
    break

# Виводимо розмірність батча (наприклад: [batch_size, канали, висота, ширина])
print(f"Форма тензора зображень (img.shape): {img.shape}")
print(f"Форма тензора міток (labels.shape): {labels.shape}")

Форма тензора зображень (img.shape): torch.Size([64, 3, 100, 100])
Форма тензора міток (labels.shape): torch.Size([64])


In [15]:
# Крок 3: Створення нейромережі
# Оскільки ми використовуємо повнозв'язну мережу (nn.Linear), зображення розміром 3 x 100 x 100 потрібно розгорнути в один вектор
# за допомогою nn.Flatten(). 
# Кількість вихідних класів у нас дорівнює кількості папок у датасеті (близько 33 класів).

from torch import nn

# Створюємо архітектуру нейромережі
model = nn.Sequential(
    nn.Flatten(),                     # Розгортаємо двовимірне зображення у вектор (3 * 100 * 100 = 30000)
    nn.Linear(30000, 128),            # Перший повнозв'язний шар
    nn.ReLU(),                        # Функція активації ReLU
    nn.Linear(128, 64),               # Прихований шар
    nn.ReLU(),
    nn.Linear(64, 32),                # Прихований шар
    nn.ReLU(),
    nn.Linear(32, len(dataset.classes)) # Вихідний шар: кількість виходів дорівнює кількості класів фруктів
)

print("Нейромережа успішно створена!")
print(model)

Нейромережа успішно створена!
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=30000, out_features=128, bias=True)
  (2): ReLU()
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Linear(in_features=64, out_features=32, bias=True)
  (6): ReLU()
  (7): Linear(in_features=32, out_features=33, bias=True)
)


In [16]:
# --- ЕТАП ТЕСТУВАННЯ БАТЧА ПЕРЕД НАВЧАННЯМ ---

# 1. Витягуємо лише один перший батч із тренувального завантажувача (X - картинки, y - мітки)
for X, y in train_loader:
    break  # Зупиняємо цикл одразу після першого батчу, щоб не перебирати весь датасет

In [17]:
# 2. Перевіряємо форму тензора зображень у батчі
# Очікуваний формат: [розмір_батча, канали, висота, ширина] (наприклад, [200, 3, 100, 100])
print("Форма тензора зображень у батчі (X.shape):")
print(X.shape)

Форма тензора зображень у батчі (X.shape):
torch.Size([64, 3, 100, 100])


In [18]:
# 3. Проводимо тестовий прогін (forward pass) одного батча через нашу нейромережу, 
# щоб переконатися відсутність помилок у розмірностях шарів
print("\nРезультат тестового прогону моделі на одному батчі:")
print(model(X))


Результат тестового прогону моделі на одному батчі:
tensor([[-0.0431,  0.0975, -0.1792,  ...,  0.1474, -0.0891, -0.1843],
        [-0.0388,  0.1052, -0.1822,  ...,  0.1324, -0.0973, -0.1865],
        [-0.0527,  0.0938, -0.1753,  ...,  0.1541, -0.0972, -0.1711],
        ...,
        [-0.0544,  0.0978, -0.1727,  ...,  0.1373, -0.1018, -0.1699],
        [-0.0672,  0.0799, -0.1660,  ...,  0.1391, -0.1073, -0.1609],
        [-0.0428,  0.0782, -0.1850,  ...,  0.1319, -0.1019, -0.1711]],
       grad_fn=<AddmmBackward0>)


In [19]:
# Крок 4: Налаштування пристрою (device) та перенесення моделі
# Визначаємо пристрій для обчислень (можна залишити "cpu" або змінити на "cuda", якщо увімкнено GPU)
device = "cuda"
print(f"Використовуваний пристрій для навчання: {device}")

# Переносимо нашу нейромережу на обраний пристрій
model = model.to(device)

Використовуваний пристрій для навчання: cuda


In [20]:
# Крок 5: Налаштування оптимізатора та функції втрат
# Для задачі класифікації використовуємо CrossEntropyLoss,
# а для оптимізації ваг — класичний алгоритм Adam із швидкістю навчання lr=0.0001

from torch.optim import Adam
from torch import nn

# Функція втрат для задач багатокласової класифікації
loss_fn = nn.CrossEntropyLoss()

# Оптимізатор Adam для оновлення параметрів моделі
optimizer = Adam(
    model.parameters(), 
    lr=0.0001
)

print("Оптимізатор та функція втрат успішно налаштовані!")

Оптимізатор та функція втрат успішно налаштовані!


In [ ]:
# Крок 6: Цикл тренування та тестування по епохах
# Створимо списки для збереження значень Loss на тренуванні та тесті, 
# а також пройдемося циклами по епохах (наприклад, поставимо поки 10 епох).

epochs = 15
list_lost_train = []
list_lost_test = []

print("🚀 Починаємо процес навчання нейромережі...")

for i in range(epochs):
    print(f"\n--- Епоха {i+1} з {epochs} ---")
    
    # --- ЕТАП ТРЕНУВАННЯ ---
    model.train()
    running_loss = 0.0
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        X_batch = x_batch.to(device)
        Y_batch = y_batch.to(device)
        
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, Y_batch)
        list_lost_train.append(loss.item())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Виводимо прогрес кожні 20 батчів, щоб бачити, що код працює
        if batch_idx % 20 == 0:
            print(f"  [Тренування] Батч {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
        
    # --- ЕТАП ТЕСТУВАННЯ ---
    model.eval()
    with torch.no_grad():
        for batch_idx, (x_batch, y_batch) in enumerate(test_loader):
            X_batch = x_batch.to(device)
            Y_batch = y_batch.to(device)
            
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, Y_batch)
            list_lost_test.append(loss.item())
            
    print(f" Епоха {i+1} успішно завершена!")

print("\n Навчання та тестування повністю завершено!")

In [ ]:
# Крок 7: Побудова графіків результатів
# Візуалізуємо динаміку зміни помилки (Loss) окремо для тренувальних та тестових даних за допомогою matplotlib

import matplotlib.pyplot as plt

# Формуємо кроки для побудови графіків
step_train = range(1, len(list_lost_train) + 1)
step_test = range(1, len(list_lost_test) + 1)

# Створюємо вікно з двома графіками поруч
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Графік тренувальної помилки
ax1.plot(step_train, list_lost_train, label='Train Loss', color='blue', marker='o')
ax1.set_title('Train Loss Performance')
ax1.set_xlabel('Steps')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# 2. Графік тестової помилки
ax2.plot(step_test, list_lost_test, label='Test Loss', color='orange', marker='s')
ax2.set_title('Test Loss Performance')
ax2.set_xlabel('Steps')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.suptitle('Loss Performance Analysis for Fruits Dataset', fontsize=14)
plt.tight_layout()
plt.show()

<h3><b>Висновок:</b></h3>
<p style="font-size: 16px; line-height: 1.5;">
У ході виконання практичної роботи було успішно побудовано, налаштовано та навчено повнозв'язну нейромережу (nn.Sequential) для задачі багатокласової класифікації фруктів. Завдяки попередньо налаштованим трансформаціям зображень (Resize, аугментаціям), використанню завантажувачів даних (DataLoader) та оптимізатора Adam на базі апаратного прискорення cuda, модель продемонструвала високу ефективність. Аналіз отриманих графіків втрат (Loss Performance) підтверджує успішність навчального процесу: значення помилки як на тренувальному, так і на тестовому наборах даних стабільно зменшуються з кожною епохою, що свідчить про відсутність сильного перенавчання (overfitting) та здатність моделі коректно узагальнювати інформацію на невідомих даних.
</p>

# **Завдання 2 (01.10.26):**
* На основі train_dataset та test_dataset з попереднього завдання створити train_loader та test_loader
* Створити нейромережу:
* використайте 3-5 шари
* перший шар Flatten
* кількість нейронів у шарах має не збільшуватись
* використайте функції активації RELU або LeakyRELU
* Збережіть kaggle notebook для подальшої роботи

In [2]:
# Крок 1: Створення DataLoader (з перевіркою) (створюємо завантажувачі даних і виводимо їхню кількість батчів)

from torch.utils.data import DataLoader

# Встановлюємо розмір батча
batch_size = 32

# Створюємо завантажувачі для тренувального та тестового наборів
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("DataLoader успішно створено!")
print(f"Кількість батчів у train_loader: {len(train_loader)}")
print(f"Кількість батчів у test_loader: {len(test_loader)}")

NameError: name 'train_dataset' is not defined

In [ ]:
# Крок 2: Візуалізація випадкових прикладів з батча 
# (Цей блок додає візуалізацію: витягує один батч і виводить сітку картинок з реальними назвами фруктів)

import matplotlib.pyplot as plt

# Витягуємо один батч даних для перевірки та візуалізації
images, labels = next(iter(train_loader))

# Створюємо сітку графіків 2x5 (10 зображень)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    # PyTorch зберігає канали на початку [C, H, W], а matplotlib чекає в кінці [H, W, C]
    img = images[i].permute(1, 2, 0)
    
    ax.imshow(img)
    # Підписуємо кожну картинку назвою її класу з датасету
    ax.set_title(dataset.classes[labels[i]])
    ax.axis('off')

plt.suptitle("Приклади зображень фруктів із тренувального батча", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Крок 3: Створення нейромережі за новими умовами
# Дотримуємося вимог: від 3 до 5 шарів, перший шар Flatten, кількість нейронів не збільшується, використовуємо ReLU
from torch import nn

# Створюємо повнозв'язну нейромережу відповідно до вимог домашньої роботи:
# - Шари поступово зменшуються (30000 -> 512 -> 128 -> 33)
# - Використовується Flatten та функції активації ReLU
model = nn.Sequential(
    nn.Flatten(),                      # 1. Розгортає зображення 3x100x100 у вектор (30000)
    nn.Linear(30000, 512),             # 2. Перший прихований шар
    nn.ReLU(),                         # Функція активації
    nn.Linear(512, 128),               # 3. Другий прихований шар (кількість зменшилась)
    nn.ReLU(),                         # Функція активації
    nn.Linear(128, len(dataset.classes)) # 4. Вихідний шар: кількість виходів дорівнює кількості класів (33)
)

print("Архітектура створеної нейромережі:")
print(model)


In [ ]:
# Крок 4: Тестовий прогін (Forward Pass)
# Переконуємося, що наша модель коректно приймає вхідний тензор і видає правильну розмірність виходу

# Пропускаємо один батч зображень через модель
output = model(images)

print(f"Розмір вхідного тензора (images.shape): {images.shape}")
print(f"Розмір вихідного тензора моделі (output.shape): {output.shape}")

# Виводимо приклад передбачення для першої картинки в батчі
prediction = output[0].argmax().item()
print(f"\nПередбачений модель клас: {dataset.classes[prediction]}")
print(f"Справжній (реальний) клас: {dataset.classes[labels[0]]}")

In [ ]:
# Крок 5: Збереження ноутбука
# Після того, як усе перевірено і запущено, зберігаємо: Save Version -> Save and Run All (або Quick Save).
# Після ПЕРЕЙМЕНОВУЄМО файл на Fruit_Classification_HW.ipynb у списку ноутбуків.